# Single-Agent Blog Writer with LangGraph + LangSmith

Enter one or more topics, generate Markdown blogs using Azure OpenAI, and observe each run in LangSmith. Run cells from top to bottom. **Use rotated API keys; never commit a notebook containing secrets.**

## Cell 1 — Install dependencies
LangGraph orchestrates the agent, `langchain-openai` connects to Azure, and LangSmith records traces.

In [1]:
%pip install -q -U langgraph langchain-openai langsmith

## Cell 2 — Configure credentials and deployment
Replace both placeholders with your **new** keys. Azure deployment must match its name in Azure AI Foundry. This cell stores secrets in notebook source, so keep the file private.

In [2]:
import os

os.environ["AZURE_OPENAI_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://xxxxxxxxxxxxxxxx.openai.azure.com/"
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-10-21"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4.1-mini"

os.environ["LANGSMITH_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "simple-blog-writer-18-Nov"

assert not os.environ["AZURE_OPENAI_API_KEY"].startswith("PASTE_"), "Set your Azure key"
assert not os.environ["LANGSMITH_API_KEY"].startswith("PASTE_"), "Set your LangSmith key"
print("Configuration loaded; keys not displayed.")

Configuration loaded; keys not displayed.


## Cell 3 — Define the single agent
The state has just two fields: the requested topic and the blog. One node calls GPT-4.1-mini and returns Markdown.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

#connect to Azure Opneai
llm = AzureChatOpenAI(
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    temperature=0.7,
)

# state variable
class BlogState(TypedDict):
    topic: str
    blog: str

# this function to bind with agent node
def blog_writer(state: BlogState):
    response = llm.invoke([
        SystemMessage(content=(
            "You are a professional blog writer. Write a clear, engaging blog "
            "in Markdown with an SEO-friendly title, introduction, useful "
            "section headings, practical examples, and conclusion. "
            "Avoid inventing facts or citations."
        )),
        HumanMessage(content=f"Write a blog about: {state['topic']}")
    ])
    return {"blog": response.content}

print("Agent ready")

Agent ready


## Cell 4 — Connect the LangGraph workflow
The graph starts, runs the blog-writing node once, and finishes. There are no additional agents or feedback loops.

In [4]:
builder = StateGraph(BlogState)
builder.add_node("blog_writer", blog_writer) # 1st parametr is -> node display name(agent), 2nd parameter -> function binded
builder.add_edge(START, "blog_writer")
builder.add_edge("blog_writer", END)
graph = builder.compile()
print("Graph ready: START → blog_writer → END")

Graph ready: START → blog_writer → END


## Cell 5 — Enter topics and generate blogs
Type comma-separated topics. Each topic creates its own graph execution and LangSmith trace. The generated Markdown is displayed below the cell.

In [5]:
from IPython.display import display, Markdown

raw_topics = input("Enter topics separated by commas: ")
topics = [t.strip() for t in raw_topics.split(",") if t.strip()]

if not topics:
    print("No topics entered. Run this cell again.")

results = {}
for topic in topics:
    print(f"\nGenerating: {topic}")
    result = graph.invoke(
        {"topic": topic, "blog": ""},
        config={
            "run_name": "blog-generation",
            "tags": ["single-agent", "blog"],
            "metadata": {"topic": topic},
        },
    )
    results[topic] = result["blog"]
    display(Markdown(result["blog"]))

Enter topics separated by commas: ai in village

Generating: ai in village


# How AI is Transforming Life in Villages: Bridging the Rural-Urban Divide

Artificial Intelligence (AI) is no longer confined to high-tech industries and urban centers. Increasingly, it is making significant inroads into rural areas, reshaping how villages function and improving the quality of life for millions. From agriculture to healthcare, education to local governance, AI is proving to be a powerful tool in bridging the rural-urban divide. This blog explores the impact of AI in villages and how it is driving sustainable development in rural communities.

## Introduction: The Promise of AI for Rural Development

Villages, often characterized by limited resources and infrastructure, face unique challenges such as access to quality education, healthcare, and efficient agricultural practices. AI technologies, with their ability to analyze data and automate processes, offer innovative solutions tailored to these challenges. By leveraging AI, villages can enhance productivity, improve services, and foster economic growth, ultimately contributing to a more balanced and inclusive development model.

## AI in Agriculture: Boosting Productivity and Sustainability

Agriculture remains the backbone of many village economies. AI-powered tools are revolutionizing farming practices by providing precise insights on soil health, weather forecasts, and crop management.

- **Smart Irrigation Systems:** AI sensors monitor soil moisture and weather conditions to optimize water usage, reducing waste and improving crop yields.
- **Crop Disease Detection:** Mobile apps using AI can identify plant diseases early through image recognition, allowing timely intervention.
- **Predictive Analytics:** AI models analyze historical data to predict pest infestations or optimal harvest times, helping farmers plan better.

These applications help farmers increase efficiency, reduce input costs, and minimize environmental impact.

## Enhancing Healthcare Access with AI

Healthcare in villages often suffers due to a shortage of medical professionals and facilities. AI bridges this gap by enabling remote diagnostics and personalized care.

- **Telemedicine and AI Chatbots:** Villagers can consult doctors via AI-enabled platforms, receiving preliminary diagnoses and health advice without traveling long distances.
- **AI-Powered Diagnostic Tools:** Portable devices equipped with AI can analyze symptoms or medical images on-site, facilitating early detection of diseases.
- **Health Monitoring:** Wearable AI devices help track vital signs, alerting healthcare providers in case of abnormalities.

Such innovations ensure timely medical attention and improve overall health outcomes in rural areas.

## Improving Education and Skill Development

Quality education is crucial for rural empowerment. AI assists in delivering personalized learning experiences and expanding access to educational resources.

- **Adaptive Learning Platforms:** AI algorithms tailor lessons according to individual student progress and learning styles.
- **Language and Literacy Tools:** AI-powered translation and speech recognition help overcome language barriers in multilingual villages.
- **Vocational Training:** AI-based simulations and virtual trainers provide skill development opportunities aligned with local economic needs.

By enhancing education, AI helps villagers acquire new skills and better employment prospects.

## Smart Village Governance and Infrastructure

AI is also transforming local governance and infrastructure management, making villages smarter and more self-sufficient.

- **Data-Driven Decision Making:** AI analyzes local data on resources, demographics, and services to help officials make informed policies.
- **Energy Management:** AI optimizes the use of renewable energy sources like solar panels, ensuring reliable power supply.
- **Waste Management:** Intelligent systems monitor waste generation and disposal, promoting cleanliness and sustainability.

These applications contribute to efficient governance and improved living standards.

## Practical Examples: AI in Action in Villages

- **AI-Powered Crop Advisory Apps:** Several rural communities use mobile apps that provide customized farming advice based on AI analysis of local conditions.
- **Remote Health Clinics:** Some villages have implemented AI-enabled kiosks offering basic health screening and connecting patients to doctors remotely.
- **Smart Water Systems:** AI-driven water pumps in certain regions automatically regulate water flow to fields, conserving resources.

These examples illustrate the tangible benefits of AI adoption in rural contexts.

## Conclusion: Embracing AI for Inclusive Rural Growth

AI holds immense potential to transform village life by addressing critical challenges in agriculture, healthcare, education, and governance. While technological adoption in rural areas faces hurdles like connectivity and digital literacy, ongoing efforts to make AI accessible and affordable are promising. Embracing AI not only empowers villagers but also fosters sustainable and inclusive development, ensuring that the benefits of the digital revolution reach every corner of the world. As AI continues to evolve, its role in shaping the future of villages will become increasingly vital.

## Cell 6 — Monitor runs in LangSmith
Open the project in LangSmith and inspect each trace. You can see the graph node, Azure model call, inputs, outputs, latency, and token information when supplied by the provider. If traces do not appear, check your LangSmith key, project, and network connectivity.

In [ ]:
from IPython.display import display, HTML
display(HTML('<a href="https://smith.langchain.com/" target="_blank">Open LangSmith → simple-blog-writer project</a>'))

### Notes
- A LangSmith project is created when traces are successfully ingested.
- The `2024-10-21` Azure API version and deployment must be supported by your Azure resource.
- Rotate the credentials shared in chat before running the notebook.
- Running a model incurs Azure OpenAI usage costs.